In [91]:
import re
import ast
from urllib.request import urlopen
import pandas as pd

In [92]:
# To do:
# - Consider automatically reformating filter names

In [98]:
"""Utilities for scraping ExoFOP time-series observations into a pandas DataFrame."""


def _extract_grid_rows(html, heading_text):
    """Extract the JavaScript row-data array for the grid beneath a named section heading.

    Parameters
    ----------
    html : str
        The full HTML page content.
    heading_text : str
        The visible heading text to locate, such as "Time Series Observations".

    Returns
    -------
    list[dict]
        A list of row dictionaries parsed from the embedded JavaScript array.
    """
    heading_match = re.search(
        r'<div[^>]*class=["\']grid_header["\'][^>]*>\s*' + re.escape(heading_text) + r'\b',
        html,
        flags=re.IGNORECASE | re.DOTALL,
    )
    if not heading_match:
        raise RuntimeError(f"Could not find section: {heading_text}")

    # Search only the content that appears after the matching heading.
    after_heading = html[heading_match.end():]
    row_match = re.search(
        r'var\s+(rowData\d+)\s*=\s*(\[[\s\S]*?\]);\s*//\s*Grid options',
        after_heading,
        flags=re.IGNORECASE | re.DOTALL,
    )
    if not row_match:
        raise RuntimeError(f"Could not find row data after section: {heading_text}")

    # Convert the JavaScript literals to Python literals before parsing.
    data = (
        row_match.group(2)
        .replace("true", "True")
        .replace("false", "False")
        .replace("null", "None")
    )

    return ast.literal_eval(data)


def get_followup_table(tic_id):
    """Fetch the follow-up observations for a given TIC ID and return a cleaned DataFrame.

    Parameters
    ----------
    tic_id : str
        TESS Input Catalog identifier, with or without the leading "TIC " prefix.

    Returns
    -------
    pandas.DataFrame
        A cleaned table with telescope, date, camera, filter, and size metadata.
    """
    if tic_id.startswith('TIC '):
        tic_id = tic_id.replace('TIC ', '')
    url = "https://exofop.ipac.caltech.edu/tess/target.php?id=" + tic_id
    with urlopen(url, timeout=20) as response:
        html = response.read().decode('utf-8', 'ignore')

    rows = _extract_grid_rows(html, 'Time Series Observations')

    df = pd.DataFrame(rows)
    df_short = df[['tstel', 'tsdate', 'tscam', 'tsfilt', 'tspix', 'tspsf', 'tspar']].copy()
    df_short.columns = ['Telescope', 'Date', 'Camera', 'Filter', r'Pix. Scale ($\arcsec$/pix)', r'PSF FWHM ($\arcsec$)', r'Aper. Rad. ($\arcsec$)']

    # Extract telescope size (m) and remove it from the Telescope column.
    df_short['Tel. Size (m)'] = df_short['Telescope'].str.extract(r'\((\d*\.?\d*)\s*m\)', expand=False).astype(float)
    df_short['Telescope'] = df_short['Telescope'].str.replace(r'\s*\(\d*\.?\d*\s*m\)', '', regex=True).str.strip()

    df_short = df_short.sort_values(by='Date', ascending=True).reset_index(drop=True)
    df_short = df_short[['Telescope', 'Tel. Size (m)', 'Date', 'Camera', 
                         'Filter', r'Pix. Scale ($\arcsec$/pix)', r'PSF FWHM ($\arcsec$)', r'Aper. Rad. ($\arcsec$)']]
    
    for filter in df_short['Filter']:
        if filter is not None:
            # Escape special LaTeX characters in the filter names
            escaped_filter = filter.replace('#', r'\#').replace('_', r'\_')
            df_short.loc[df_short['Filter'] == filter, 'Filter'] = escaped_filter
    
    # Truncate trailing zeros and convert floats to strings for LaTeX formatting
    float_columns = ['Tel. Size (m)', r'Pix. Scale ($\arcsec$/pix)', r'PSF FWHM ($\arcsec$)', r'Aper. Rad. ($\arcsec$)']
    for col in float_columns:
        df_short[col] = df_short[col].apply(lambda x: ('{:.3f}'.format(x)).rstrip('0').rstrip('.') if pd.notnull(x) else x)

    # Replace NaNs with '---'
    df_short = df_short.fillna('---')

    return df_short


def generate_master_table(tic_list, toi_list):
    """Build a combined follow-up table for many TIC/TOI pairs.

    Parameters
    ----------
    tic_list : list[str]
        TIC identifiers, with or without the "TIC " prefix.
    toi_list : list[str]
        TOI identifiers, with or without the "TOI-" prefix.

    Returns
    -------
    pandas.DataFrame
        A concatenated table with TIC and TOI columns added to each row group.
    """
    master_df = pd.DataFrame()

    for tic_id, toi_id in zip(tic_list, toi_list):
        if not tic_id.startswith('TIC '):
            tic_id = 'TIC ' + tic_id

        if toi_id.startswith('TOI-'):
            toi_id = toi_id.replace('TOI-', '')
        try:
            df = get_followup_table(tic_id)
        except Exception as e:
            print(f"Error occurred while fetching follow-up table for TOI-{toi_id}: {e}")
            continue

        # Add TIC and TOI ids to the first row for this target.
        df['TIC ID'] = tic_id.replace('TIC ', '')
        df.loc[1:, 'TIC ID'] = ''
        df['TOI Number'] = toi_id
        df.loc[1:, 'TOI Number'] = ''

        master_df = pd.concat([master_df, df], ignore_index=True)
        master_df = master_df[['TIC ID', 'TOI Number', 'Telescope', 'Tel. Size (m)', 'Date', 'Camera',
                                'Filter', r'Pix. Scale ($\arcsec$/pix)', r'PSF FWHM ($\arcsec$)', r'Aper. Rad. ($\arcsec$)']]
    return master_df

def convert_to_latex_and_save(df, filename):
    """Convert a DataFrame to LaTeX format and save it to a file.

    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame to convert.
    filename : str
        The name of the file to save the LaTeX output.
    """
    latex_str = df.to_latex(index=False, escape=False, caption='Summary of Follow-up Observations', label='tab:followup', longtable=True)
    with open(filename, 'w') as f:
        f.write(latex_str)

def create_and_save_followup_table(tic_list, toi_list, output_filename):
    """Generate the follow-up table and save it as a LaTeX file.

    Parameters
    ----------
    tic_list : list[str]
        List of TIC identifiers.
    toi_list : list[str]
        List of TOI identifiers.
    output_filename : str
        The filename for the output LaTeX file.
    """
    master_df = generate_master_table(tic_list, toi_list)
    convert_to_latex_and_save(master_df, output_filename)

In [99]:
generate_master_table(['TIC 371573539'], ['TOI-6148'])

,TIC ID,TOI Number,Telescope,Tel. Size (m),Date,Camera,Filter,Pix. Scale ($\arcsec$/pix),PSF FWHM ($\arcsec$),Aper. Rad. ($\arcsec$)
0,371573539,6148,TCS-MuSCAT2,1.52,2023-08-02,MuSCAT2,"g, r, i, z\_s",0.44,---,---
1,,,LCO-Teid-1m0,1,2024-11-04,SINISTRO,ip,0.389,1.14,6
2,,,LCO-HAL-0m35,0.35,2025-06-08,QHY600,gp,0.73,2.13,7


In [100]:
tic_ids = ['371864043', '34297761', '191202679', '312811620', '284206913', '452938965', '320281287', '368734712', '444558604', '302381397'
, '400103802', '313194972', '193765661', '86898676', '67444896', '345193111', '371573539', '190822775', '67478724', '197743152'
, '257207557', '326475995', '63718617', '95191643', '125563127', '242674266', '125695940', '352409708', '285592400', '159084486']
toi_ids = ['3041', '3365', '3601', '3788', '3972', '3988', '3998', '4009', '4079', '4088', '4140', '4144', '5236', '5432', '5479', '5925',
            '6148', '6166', '6171', '6191', '6184', '6208', '6334', '6417', '6443', '7219', '7266', '7404', '7425', '7574']
output_filename = 'followup_table.tex'

create_and_save_followup_table(tic_ids, toi_ids, output_filename)

<unknown>:1: SyntaxWarning: invalid escape sequence '\/'


Error occurred while fetching follow-up table for TOI-5236: "None of [Index(['tstel', 'tsdate', 'tscam', 'tsfilt', 'tspix', 'tspsf', 'tspar'], dtype='object')] are in the [columns]"
